# “How well does it run?” — production, value, and efficiency

This notebook answers a companion question for current healthy campaign partitions:

> **What did successful Runs produce, what operational value was measured, and what did those Runs cost?**

It first invokes the shared `does-it-run@1.0.0` runtime as a gate. Failed, unknown, unobserved, and orchestrator-gated partitions are skipped before detailed Issue, Audit, Tool, or Domain evidence is loaded. It then invokes `how-well-does-it-run@1.0.0` for the remaining `yes` partitions and recoverable `running` partitions with a successful boundary.

The measure keeps three axes separate:

1. **Production** — distinct safe outputs created by successful Runs.
2. **Value measurement** — native campaign-specific operational-value grader results.
3. **Efficiency** — AIC and duration context.

It does not invent a composite quality score. Output creation does not prove acceptance or value, a grader error is not zero value, and lower cost does not itself mean higher value.

> This notebook reads only canonical bounded evidence. Raw prompts, transcripts, tool arguments, response bodies, artifact bodies, and complete Actions logs are not loaded.


In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import json
import re
import subprocess
import time

import pandas as pd
from IPython.display import HTML, Markdown, display

ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'activity/cao.mjs').exists()),
    None,
)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the gh-aw-cao repository')

SHARDS = ROOT / '.cao/gh-aw-logs-shards'
RUNTIME_BRIDGE = ROOT / 'research/computations/compute-does-it-run.mjs'
REFRESH_DATA = True
MAX_DISPLAY_ROWS = 100

pd.set_option('display.max_colwidth', 90)
print('Repository:', ROOT)


Repository: /Users/mnkiefer/gh-aw-cao


## 1. Establish the evidence boundary

The Activity snapshot contains both raw workflow Runs and Runs enriched by gh-aw. The `does-it-run` measure needs enriched `kind: run` records to associate workflow evidence with campaign semantics.

`audit-jsonl` reports coverage before any health conclusion is made. Partial evidence does not become a healthy or failed answer by default.


In [2]:
def cao(*arguments):
    completed = subprocess.run(
        ['node', 'activity/cao.mjs', *arguments],
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    return json.loads(completed.stdout)

if REFRESH_DATA:
    download = cao('download')

audit = cao('audit-jsonl')
source = audit['source']
canonical = audit['canonical']
evidence_boundary = pd.DataFrame([{
    'shards': audit['shards'],
    'unique raw Runs': source['uniqueRawRuns'],
    'unique enriched Runs': source['uniqueEnrichedRuns'],
    'enrichment coverage %': source['enrichmentCoveragePercent'],
    'canonical Runs': canonical['runs'],
    'canonical Audits': canonical['audits'],
}])

display(evidence_boundary)
print('Every result below is limited to enriched kind:run evidence.')


,shards,unique raw Runs,unique enriched Runs,enrichment coverage %,canonical Runs,canonical Audits
0,6,31951,2915,9.1,31951,85891


Every result below is limited to enriched kind:run evidence.


## 2. Resolve campaign identity and expected targets

The raw data tells us what was observed. Checked-in campaign records and control policy tell us what was expected.

Resolution rules:

1. workflow identity and role come from each campaign’s `cao.json`;
2. explicit policy `targets` take precedence;
3. a review-mode campaign without explicit targets expects the CAO control repository (`githubnext/gh-aw-cao`); and
4. another observed repository is retained as `observed-extra`, but it does not determine configured campaign health.

This prevents observed activity from silently widening rollout policy.


In [3]:
registry = []
for campaign_file in sorted(ROOT.glob('*/cao.json')):
    definition = json.loads(campaign_file.read_text())
    campaign = definition['campaign']
    orchestrator = definition['orchestrator']
    registry.append({
        'campaign': campaign,
        'role': 'orchestrator',
        'workflow': orchestrator,
        'workflow_path': f'.github/workflows/{orchestrator}.lock.yml',
    })
    for worker_role, workflow in definition.get('workers', {}).items():
        registry.append({
            'campaign': campaign,
            'role': 'worker',
            'worker_role': worker_role,
            'workflow': workflow,
            'workflow_path': f'.github/workflows/{workflow}.lock.yml',
        })

registry_df = pd.DataFrame(registry)
workflow_by_path = registry_df.set_index('workflow_path').to_dict('index')

policy = json.loads((ROOT / '.github/workflows/cao.json').read_text())
configurations = policy.get('control-plane', {}).get('campaigns', {})
control_repository = 'githubnext/gh-aw-cao'
expected_targets = {}
target_resolution = {}

for campaign in sorted(registry_df.campaign.unique()):
    raw_configuration = configurations.get(campaign, {})
    configuration = raw_configuration if isinstance(raw_configuration, dict) else {}
    explicit_targets = set((configuration.get('targets') or {}).keys())
    if explicit_targets:
        expected_targets[campaign] = explicit_targets
        target_resolution[campaign] = 'explicit-policy'
    elif configuration.get('mode', 'review') == 'review':
        expected_targets[campaign] = {control_repository}
        target_resolution[campaign] = 'review-control-repository'
    else:
        expected_targets[campaign] = set()
        target_resolution[campaign] = 'unavailable'

inventory = registry_df.groupby(['campaign', 'role']).size().unstack(fill_value=0).reset_index()
inventory['expected targets'] = inventory.campaign.map(lambda value: len(expected_targets[value]))
inventory['target resolution'] = inventory.campaign.map(target_resolution)
display(inventory)


role,campaign,orchestrator,worker,expected targets,target resolution
0,cao-evolution,1,6,1,review-control-repository
1,dependabot,1,1,1,explicit-policy
2,eslint-rules,1,5,1,review-control-repository
3,eu-cra-compliance,1,6,1,review-control-repository
4,optimization,1,7,1,review-control-repository
5,repo-assist,1,4,1,review-control-repository
6,self-care,1,15,1,explicit-policy
7,software-development-practices,1,2,1,review-control-repository
8,uk-ai-advisory,1,1,1,review-control-repository


## 3. Read and deduplicate enriched Runs

Each JSONL shard is read once. The notebook keeps only fields needed to identify a Run, order it, evaluate its state, group failures, and link back to evidence.

Worker target names are extracted from producer titles shaped as `… · OWNER/REPOSITORY · MODE`. A missing target remains unknown rather than being assigned to an observed repository.


In [4]:
target_pattern = re.compile(r'^[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+$')


def target_from_name(name):
    parts = [part.strip() for part in (name or '').split('·')]
    target = next((part for part in parts[1:] if target_pattern.fullmatch(part)), None)
    mode = next(
        (part.lower() for part in reversed(parts[1:]) if part.lower() in {'live', 'review', 'preview'}),
        None,
    )
    return target, mode


def run_date(run):
    return run.get('started_at') or run.get('created_at') or run.get('updated_at')


started_at = time.perf_counter()
lines_read = 0
run_observations = 0
matched_observations = 0
deduplicated = {}

for shard in sorted(SHARDS.glob('*.jsonl')):
    with shard.open('rb') as handle:
        for line in handle:
            lines_read += 1
            if b'"kind":"run"' not in line[:80]:
                continue
            run_observations += 1
            run = json.loads(line).get('run') or {}
            workflow = workflow_by_path.get(run.get('workflow_path'))
            if workflow is None:
                continue
            matched_observations += 1
            target, mode = target_from_name(run.get('workflow_name')) if workflow['role'] == 'worker' else (None, None)
            row = {
                **workflow,
                'execution_repository': run.get('repository'),
                'target_repository': target,
                'rollout_mode': mode,
                'run_id': run.get('run_id'),
                'attempt': int(run.get('run_attempt') or 1),
                'status': (run.get('status') or 'unknown').lower(),
                'conclusion': (run.get('conclusion') or '').lower() or None,
                'classification': run.get('classification'),
                'failure_kind': run.get('failure_kind'),
                'run_date': run_date(run),
                'updated_at': run.get('updated_at'),
                'url': run.get('url'),
            }
            key = (row['execution_repository'], row['run_id'], row['attempt'])
            previous = deduplicated.get(key)
            if previous is None or (row['updated_at'] or '') >= (previous['updated_at'] or ''):
                deduplicated[key] = row

runs = pd.DataFrame(deduplicated.values())
for field in ('run_date', 'updated_at'):
    runs[field] = pd.to_datetime(runs[field], utc=True, errors='coerce')

load_metrics = pd.DataFrame([{
    'JSONL lines read': lines_read,
    'enriched Run observations': run_observations,
    'campaign Run observations': matched_observations,
    'distinct campaign Run attempts': len(runs),
    'declared workflows observed': runs.workflow.nunique(),
    'load seconds': round(time.perf_counter() - started_at, 2),
}])

display(load_metrics)
display(runs.groupby(['campaign', 'role']).size().rename('distinct Runs').reset_index())


,JSONL lines read,enriched Run observations,campaign Run observations,distinct campaign Run attempts,declared workflows observed,load seconds
0,146091,74024,57762,2213,45,5.34


,campaign,role,distinct Runs
0,cao-evolution,orchestrator,17
1,cao-evolution,worker,70
2,dependabot,orchestrator,84
3,dependabot,worker,64
4,eslint-rules,orchestrator,8
5,eslint-rules,worker,6
6,eu-cra-compliance,orchestrator,17
7,eu-cra-compliance,worker,369
8,optimization,orchestrator,97
9,optimization,worker,980


## 4. Establish the current runtime-health gate

The notebook prepares the same orchestrator and worker-target partitions as `does-it-run`. The shared runtime evaluates orchestrators first and isolates each worker target. Only current `yes` partitions, or `running` partitions with a previous successful boundary, become eligible for the more expensive value-evidence join below.

This gate prevents historical successful Runs from distracting from a current failure and avoids loading detailed records for partitions whose next step is diagnosis or evidence restoration.


In [5]:
def json_scalar(value):
    if value is None or (not isinstance(value, (list, dict)) and pd.isna(value)):
        return None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value.item() if hasattr(value, 'item') else value


def runtime_run(row):
    return {
        'id': f"github:run:{row['run_id']}:attempt:{row['attempt']}",
        'githubRunId': json_scalar(row['run_id']),
        'attempt': int(row['attempt']),
        'status': row['status'],
        'conclusion': json_scalar(row['conclusion']),
        'classification': json_scalar(row['classification']),
        'failureKind': json_scalar(row['failure_kind']),
        'startedAt': json_scalar(row['run_date']),
        'updatedAt': json_scalar(row['updated_at']),
        'runLink': json_scalar(row['url']),
    }


def make_partition(campaign, workflow, role, frame, target=None, membership=None):
    ordered = frame.sort_values(
        ['run_date', 'run_id', 'attempt'],
        ascending=[False, False, False],
        na_position='last',
    )
    return {
        'campaignId': campaign,
        'workflowId': workflow,
        'workflowRole': role,
        'workflowState': 'active',
        'targetRepositoryId': target,
        'targetScopeMembership': membership,
        'orderedNewestFirst': True,
        'runs': [runtime_run(row) for row in ordered.to_dict('records')],
    }


campaign_inputs = []
for campaign in sorted(registry_df.campaign.unique()):
    definitions = registry_df[registry_df.campaign == campaign]
    orchestrator_partitions = []
    worker_partitions = []

    for definition in definitions[definitions.role == 'orchestrator'].to_dict('records'):
        matching = runs[(runs.campaign == campaign) & (runs.workflow == definition['workflow'])]
        orchestrator_partitions.append(make_partition(
            campaign,
            definition['workflow'],
            'orchestrator',
            matching,
        ))

    expected = expected_targets[campaign]
    for definition in definitions[definitions.role == 'worker'].to_dict('records'):
        matching = runs[(runs.campaign == campaign) & (runs.workflow == definition['workflow'])]
        observed = set(matching.target_repository.dropna().unique())
        targets = sorted(expected | observed)
        if matching.target_repository.isna().any() or not targets:
            targets.append(None)
        for target in targets:
            target_runs = (
                matching[matching.target_repository == target]
                if target is not None
                else matching[matching.target_repository.isna()]
            )
            membership = 'expected' if target in expected else 'observed-extra' if expected else 'unknown'
            worker_partitions.append(make_partition(
                campaign,
                definition['workflow'],
                'worker',
                target_runs,
                target,
                membership,
            ))

    campaign_inputs.append({
        'campaignId': campaign,
        'orchestratorPartitions': orchestrator_partitions,
        'workerPartitions': worker_partitions,
    })

runtime_request = {'campaigns': campaign_inputs, 'benchmarkIterations': 30}
completed = subprocess.run(
    ['node', str(RUNTIME_BRIDGE)],
    cwd=ROOT,
    input=json.dumps(runtime_request),
    text=True,
    capture_output=True,
    check=True,
)
runtime_response = json.loads(completed.stdout)
runtime_result = runtime_response['result']
runtime_benchmark = runtime_response['benchmark']

partition_rows = []
for result in runtime_result['partitionResults']:
    latest_run = result.get('latestRun') or {}
    latest_success = result.get('latestSuccess') or {}
    partition_rows.append({
        'campaign': result['campaignId'],
        'workflow': result['workflowId'],
        'role': result['workflowRole'],
        'target_repository': result.get('targetRepositoryId'),
        'target_scope_membership': result.get('targetScopeMembership'),
        'answer': result['answer'],
        'needs_attention': result['needsAttention'],
        'latest_run_id': latest_run.get('githubRunId'),
        'latest_run_at': latest_run.get('observedAt'),
        'latest_run_url': latest_run.get('url'),
        'latest_success_id': latest_success.get('githubRunId'),
        'latest_success_at': latest_success.get('observedAt'),
        'runs_since_success': result['runsSinceSuccess'],
        'terminal_non_successes': result['terminalNonSuccessesSinceSuccess'],
        'active_runs': result['activeRunsSinceSuccess'],
        'unknown_date_runs': result['unknownDateRunCount'],
        'records_visited': result['recordsVisited'],
        'partition_runs_retained': result['partitionRunCount'],
    })
partitions_df = pd.DataFrame(partition_rows)

error_rows = []
for group in runtime_result['errorGroups']:
    latest_reference = group['runReferences'][0] if group['runReferences'] else {}
    error_rows.append({
        'campaign': group['campaignId'],
        'workflow': group['workflowId'],
        'role': group['workflowRole'],
        'target_repository': group.get('targetRepositoryId'),
        'target_scope_membership': group.get('targetScopeMembership'),
        'error_key': group['errorKey'],
        'count': group['count'],
        'latest_observed_at': group['latestObservedAt'],
        'latest_run_id': latest_reference.get('githubRunId'),
        'latest_run_url': latest_reference.get('url'),
        'current_run_ids': [reference.get('githubRunId') for reference in group['runReferences']],
        'current_run_urls': [reference.get('url') for reference in group['runReferences']],
    })
errors_df = pd.DataFrame(error_rows)

campaign_rows = []
for result in runtime_result['campaignResults']:
    campaign = result['campaignId']
    orchestrators = partitions_df[
        (partitions_df.campaign == campaign) & (partitions_df.role == 'orchestrator')
    ]
    campaign_rows.append({
        'campaign': campaign,
        'answer': result['answer'],
        'worker_evaluation_state': result['workerEvaluationState'],
        'orchestrator_answer': orchestrators.answer.iloc[0] if len(orchestrators) else 'unknown',
        'worker_partitions_evaluated': result['workerPartitionCount'],
        'attention_partitions': result['attentionPartitionCount'],
        'explicit_target_count': len(expected_targets[campaign]),
        'target_coverage': 'complete' if expected_targets[campaign] else 'unavailable',
        'records_visited': result['recordsVisited'],
        'partition_runs_retained': result['partitionRunCount'],
        'worker_partitions_skipped_by_gate': result['skippedWorkerPartitionCount'],
        'worker_runs_skipped_by_gate': result['skippedWorkerRunCount'],
    })
campaigns_df = pd.DataFrame(campaign_rows)


## 5. “How well does it run?” — inspect successful Runs

The default path first gates on `does-it-run`: failed, unknown, unobserved, and orchestrator-gated partitions go to diagnosis and are skipped here. Runtime success only says that a workflow completed. This companion measure reuses the same Campaign, Workflow, and target Repository partitions and looks at successful Runs within the detailed-record retention window.

It reports three independent axes:

1. **Production** — did the Runs create distinct safe outputs such as issues, pull requests, updates, or comments?
2. **Value measurement** — did a campaign-specific operational-value grader produce a finite native value?
3. **Efficiency** — what AIC and duration did those successful Runs consume?

Important interpretation rules:

- A created output is **produced**, not necessarily accepted, useful, implemented, or valuable.
- A passing grader means value was measured. A measured value of `0` remains zero; it is not converted to success or failure.
- A grader error means value is unknown, not zero.
- AIC and duration describe cost, not value.
- Orchestrator outputs are shown separately from worker-target outputs because coordination activity is not target operational value.


In [6]:
HOW_WELL_BRIDGE = ROOT / 'research/computations/compute-how-well-does-it-run.mjs'
DETAIL_RETENTION_DAYS = 30

canonical_successes = cao(
    'query',
    '--collection',
    'runs',
    '--where',
    'conclusion=success',
    '--limit',
    '50000',
)
value_registry = {
    f".github/workflows/{row['workflow']}.md": row
    for row in registry_df.to_dict('records')
}
all_campaign_successes = [
    run for run in canonical_successes
    if run.get('workflowPath') in value_registry
]

success_times = pd.to_datetime([
    run.get('startedAt') or run.get('createdAt') or run.get('updatedAt')
    for run in all_campaign_successes
], utc=True, errors='coerce')
evidence_window_end = success_times.max()
evidence_window_start = evidence_window_end - pd.Timedelta(days=DETAIL_RETENTION_DAYS)

all_campaign_successes = [
    run for run in all_campaign_successes
    if pd.notna(pd.to_datetime(
        run.get('startedAt') or run.get('createdAt') or run.get('updatedAt'),
        utc=True,
        errors='coerce',
    ))
    and pd.to_datetime(
        run.get('startedAt') or run.get('createdAt') or run.get('updatedAt'),
        utc=True,
    ) >= evidence_window_start
]


def normalized_target(value):
    return None if value is None or pd.isna(value) else value


eligible_runtime_partitions = {}
for row in partitions_df.to_dict('records'):
    has_success_boundary = pd.notna(row.get('latest_success_id'))
    if row['answer'] == 'yes' or (row['answer'] == 'running' and has_success_boundary):
        eligible_runtime_partitions[(
            row['campaign'],
            row['workflow'],
            row['role'],
            normalized_target(row.get('target_repository')),
        )] = row['answer']


def value_partition_key(run):
    definition = value_registry[run['workflowPath']]
    target = run.get('targetRepository') if definition['role'] == 'worker' else None
    return (definition['campaign'], definition['workflow'], definition['role'], target)


all_success_partition_keys = {value_partition_key(run) for run in all_campaign_successes}
campaign_successes = [
    run for run in all_campaign_successes
    if value_partition_key(run) in eligible_runtime_partitions
]
skipped_value_partition_count = len(all_success_partition_keys - set(eligible_runtime_partitions))
skipped_successful_run_count = len(all_campaign_successes) - len(campaign_successes)

canonical_issues = cao('query', '--collection', 'issues', '--limit', '100000')
success_ids = {run['id'] for run in campaign_successes}
issues_by_run = defaultdict(list)
for issue in canonical_issues:
    if issue.get('runId') in success_ids:
        issues_by_run[issue['runId']].append(issue)


def operational_value_results(run):
    return [
        result
        for result in (run.get('graders') or {}).get('results', [])
        if result.get('id') == 'operational-value'
        or result.get('source') == 'operational-value'
    ]


def value_runtime_run(run):
    return {
        'id': run['id'],
        'githubRunId': run.get('githubRunId'),
        'conclusion': run.get('conclusion'),
        'observedAt': run.get('startedAt') or run.get('createdAt') or run.get('updatedAt'),
        'runLink': run.get('runLink'),
        'safeItemsCount': run.get('safeItemsCount'),
        'outputs': issues_by_run[run['id']],
        'operationalValueResults': operational_value_results(run),
        'aic': run.get('aic'),
        'durationSeconds': run.get('agenticDurationSeconds'),
    }


grouped_successes = defaultdict(list)
for run in campaign_successes:
    definition = value_registry[run['workflowPath']]
    target = run.get('targetRepository') if definition['role'] == 'worker' else None
    grouped_successes[(
        definition['campaign'],
        definition['workflow'],
        definition['role'],
        target,
    )].append(run)

value_partitions = []
for (campaign, workflow, role, target), successful_runs in sorted(
    grouped_successes.items(),
    key=lambda item: tuple('' if value is None else str(value) for value in item[0]),
):
    expected = expected_targets[campaign]
    membership = (
        'expected' if target in expected
        else 'observed-extra' if target is not None and expected
        else 'unknown'
    ) if role == 'worker' else None
    value_partitions.append({
        'campaignId': campaign,
        'workflowId': workflow,
        'workflowRole': role,
        'targetRepositoryId': target,
        'targetScopeMembership': membership,
        'runtimeAnswer': eligible_runtime_partitions[(campaign, workflow, role, target)],
        'evidenceWindowStart': evidence_window_start.isoformat(),
        'evidenceWindowEnd': evidence_window_end.isoformat(),
        'successfulRuns': [value_runtime_run(run) for run in successful_runs],
    })

value_request = {'partitions': value_partitions}
completed = subprocess.run(
    ['node', str(HOW_WELL_BRIDGE)],
    cwd=ROOT,
    input=json.dumps(value_request),
    text=True,
    capture_output=True,
    check=True,
)
value_response = json.loads(completed.stdout)
value_result = value_response['result']

value_rows = []
for result in value_result['partitionResults']:
    measured_values = result['measuredOperationalValues']
    value_rows.append({
        'campaign': result['campaignId'],
        'workflow': result['workflowId'],
        'role': result['workflowRole'],
        'target_repository': result.get('targetRepositoryId'),
        'target_scope_membership': result.get('targetScopeMembership'),
        'successful_runs': result['successfulRunCount'],
        'production': result['productionState'],
        'output_evidence_coverage_%': round(100 * (result['outputEvidenceCoverage'] or 0), 1),
        'producer_reported_outputs': result['producerReportedOutputCount'],
        'retained_distinct_outputs': result['distinctOutputCount'],
        'output_kinds': ', '.join(
            f'{kind}={count}' for kind, count in result['outputKinds'].items()
        ) or None,
        'value_measurement': result['valueMeasurementState'],
        'measured_value_count': len(measured_values) + result['omittedMeasuredOperationalValueCount'],
        'measured_values': ', '.join(
            f"{entry['value']} {entry.get('unit') or ''}".strip()
            for entry in measured_values[:5]
        ) or None,
        'grader_pass': result['operationalValueStatusCounts']['pass'],
        'grader_error': result['operationalValueStatusCounts']['error'],
        'grader_unavailable': result['operationalValueStatusCounts']['unavailable'],
        'aic_measured_runs': result['aic']['measuredRunCount'],
        'total_aic': result['aic']['total'],
        'median_aic': result['aic']['median'],
        'duration_measured_runs': result['durationSeconds']['measuredRunCount'],
        'median_duration_minutes': (
            round(result['durationSeconds']['median'] / 60, 1)
            if result['durationSeconds']['median'] is not None
            else None
        ),
    })
value_partitions_df = pd.DataFrame(value_rows)

value_output_rows = []
value_metric_rows = []
for result in value_result['partitionResults']:
    identity = {
        'campaign': result['campaignId'],
        'workflow': result['workflowId'],
        'role': result['workflowRole'],
        'target_repository': result.get('targetRepositoryId'),
    }
    for output in result['outputReferences']:
        value_output_rows.append({**identity, **output})
    for metric in result['measuredOperationalValues']:
        value_metric_rows.append({**identity, **metric})
value_outputs_df = pd.DataFrame(value_output_rows)
value_metrics_df = pd.DataFrame(value_metric_rows)

print(
    f"Successful-Run evidence window: {evidence_window_start.isoformat()} "
    f"through {evidence_window_end.isoformat()}"
)
print(
    f"Skipped {skipped_value_partition_count} currently failed, unknown, unobserved, or gated partition(s) "
    f"and {skipped_successful_run_count} historical successful Run(s) before detailed value evaluation."
)
print(
    f"{value_result['totals']['successfulRunCount']} successful Runs across "
    f"{value_result['totals']['partitionCount']} partitions; "
    f"{value_result['totals']['distinctOutputCount']} retained distinct output observations; "
    f"{value_result['totals']['measuredOperationalValueCount']} measured operational-value results."
)


Successful-Run evidence window: 2026-08-23T02:39:08+00:00 through 2026-09-22T02:39:08+00:00
Skipped 63 currently failed, unknown, unobserved, or gated partition(s) and 505 historical successful Run(s) before detailed value evaluation.
1209 successful Runs across 57 partitions; 1083 retained distinct output observations; 174 measured operational-value results.


In [7]:
def display_value_table(frame):
    html = frame.to_html(escape=True)
    for value in ('produced', 'measured'):
        html = html.replace(
            f'<td>{value}</td>',
            f'<td style="background-color:#d1fae5;color:#065f46;font-weight:700">{value}</td>',
        )
    for value in ('evaluation-error',):
        html = html.replace(
            f'<td>{value}</td>',
            f'<td style="background-color:#fee2e2;color:#991b1b;font-weight:700">{value}</td>',
        )
    for value in ('unknown', 'unavailable', 'not-configured', 'none-observed'):
        html = html.replace(
            f'<td>{value}</td>',
            f'<td style="background-color:#fef3c7;color:#92400e;font-weight:700">{value}</td>',
        )
    display(HTML(html))


worker_value_df = value_partitions_df[value_partitions_df.role == 'worker'].sort_values(
    ['campaign', 'workflow', 'target_repository'],
    na_position='last',
)
display(Markdown('### Worker-target successful-Run evidence'))
display_value_table(worker_value_df)

display(Markdown('### Evidence coverage summary'))
coverage_summary = pd.DataFrame([{
    'successful Runs': int(value_partitions_df.successful_runs.sum()),
    'successful worker Runs': int(worker_value_df.successful_runs.sum()),
    'partitions with produced outputs': int((worker_value_df.production == 'produced').sum()),
    'retained distinct outputs': int(worker_value_df.retained_distinct_outputs.sum()),
    'partitions with measured operational value': int((worker_value_df.value_measurement == 'measured').sum()),
    'partitions with grader errors': int((worker_value_df.value_measurement == 'evaluation-error').sum()),
    'partitions without configured value measurement': int((worker_value_df.value_measurement == 'not-configured').sum()),
}])
display(coverage_summary)


def value_drilldown(campaign):
    known = set(value_partitions_df.campaign)
    if campaign not in known:
        raise KeyError(f'Choose from {sorted(known)}')

    display(Markdown(f'## Successful-Run value evidence: `{campaign}`'))
    campaign_partitions = value_partitions_df[
        value_partitions_df.campaign == campaign
    ].sort_values(['role', 'workflow', 'target_repository'], na_position='last')
    display_value_table(campaign_partitions)

    campaign_outputs = (
        value_outputs_df[value_outputs_df.campaign == campaign]
        if not value_outputs_df.empty
        else pd.DataFrame()
    )
    display(Markdown('### What successful Runs produced'))
    display(
        campaign_outputs
        if not campaign_outputs.empty
        else Markdown('No retained detailed output references are available.')
    )

    campaign_metrics = (
        value_metrics_df[value_metrics_df.campaign == campaign]
        if not value_metrics_df.empty
        else pd.DataFrame()
    )
    display(Markdown('### Native operational-value measurements'))
    display(
        campaign_metrics
        if not campaign_metrics.empty
        else Markdown(
            'No finite operational-value measurement is available. Inspect the '
            '`value_measurement`, `grader_error`, and `grader_unavailable` columns above.'
        )
    )

    display(Markdown('### Interpretation'))
    produced = int((campaign_partitions.production == 'produced').sum())
    measured = int((campaign_partitions.value_measurement == 'measured').sum())
    grader_errors = int((campaign_partitions.value_measurement == 'evaluation-error').sum())
    display(Markdown(
        f'- **{produced}** partition(s) produced output evidence.\n'
        f'- **{measured}** partition(s) have finite native operational-value measurements.\n'
        f'- **{grader_errors}** partition(s) have value-evaluation errors.\n'
        '- Produced outputs remain **unverified production** until acceptance or disposition evidence is available.\n'
        '- Use AIC and duration only as efficiency context; do not interpret lower cost as higher value by itself.'
    ))


value_drilldown('dependabot')


### Worker-target successful-Run evidence

,campaign,workflow,role,target_repository,target_scope_membership,successful_runs,production,output_evidence_coverage_%,producer_reported_outputs,retained_distinct_outputs,output_kinds,value_measurement,measured_value_count,measured_values,grader_pass,grader_error,grader_unavailable,aic_measured_runs,total_aic,median_aic,duration_measured_runs,median_duration_minutes
1,cao-evolution,cao-evolution-catalog-advisor,worker,githubnext/gh-aw-cao,expected,9,none-observed,100,0,0,NaN,not-configured,0,NaN,0,0,0,7,481.69703,66.018460,7,10.4
2,cao-evolution,cao-evolution-compiler-security,worker,githubnext/gh-aw-cao,expected,8,produced,100,7,7,create_issue=7,evaluation-error,0,NaN,0,7,0,7,594.64303,52.116140,7,21.6
3,cao-evolution,cao-evolution-efficiency,worker,githubnext/gh-aw-cao,expected,18,produced,100,1,0,NaN,not-configured,0,NaN,0,0,0,16,854.85907,41.908300,16,9.1
4,cao-evolution,cao-evolution-failures-investigator,worker,githubnext/gh-aw-cao,expected,9,none-observed,100,0,0,NaN,evaluation-error,0,NaN,0,7,0,7,115.93225,15.634570,7,6.6
5,cao-evolution,cao-evolution-integrity,worker,githubnext/gh-aw-cao,expected,18,produced,100,2,0,NaN,not-configured,0,NaN,0,0,0,16,1423.01872,71.057425,16,9.4
6,cao-evolution,cao-evolution-reliability,worker,githubnext/gh-aw-cao,expected,19,produced,100,15,20,create_issue=20,not-configured,0,NaN,0,0,0,17,2828.61938,167.098810,17,16.6
8,dependabot,dependabot-update-planner,worker,github/gh-aw-firewall,observed-extra,2,produced,100,5,5,create_issue=5,evaluation-error,0,NaN,0,2,0,2,186.62538,93.312690,2,12.1
10,eslint-rules,eslint-rules-miner,worker,github/gh-aw-firewall,observed-extra,9,none-observed,100,0,0,NaN,not-configured,0,NaN,0,0,0,6,617.73163,99.811575,6,13.5
12,optimization,optimization-agents-md-curator,worker,github/gh-aw,observed-extra,84,produced,100,51,64,create_issue=64,measured,34,"0 ratio, 1 ratio, 1 ratio, 1 ratio, 0 ratio",34,23,24,81,7504.09264,98.135550,81,13.3
13,optimization,optimization-agents-md-curator,worker,github/gh-aw-mcpg,observed-extra,31,produced,100,21,21,create_issue=21,measured,13,"1 ratio, 1 ratio, 1 ratio, 0 ratio, 0 ratio",13,8,8,29,5121.66783,210.231720,29,17.2


### Evidence coverage summary

,successful Runs,successful worker Runs,partitions with produced outputs,retained distinct outputs,partitions with measured operational value,partitions with grader errors,partitions without configured value measurement
0,1209,960,41,1083,11,11,25


## Successful-Run value evidence: `dependabot`

...

,campaign,workflow,role,target_repository,id,type,status,url,observedAt
27,dependabot,dependabot-update-planner,worker,github/gh-aw-firewall,https://github.com/githubnext/gh-aw-cao/issues/13321,create_issue,created,https://github.com/githubnext/gh-aw-cao/issues/13321,2026-09-22T01:50:55.000Z
28,dependabot,dependabot-update-planner,worker,github/gh-aw-firewall,https://github.com/githubnext/gh-aw-cao/issues/13319,create_issue,created,https://github.com/githubnext/gh-aw-cao/issues/13319,2026-09-22T01:50:55.000Z
29,dependabot,dependabot-update-planner,worker,github/gh-aw-firewall,https://github.com/githubnext/gh-aw-cao/issues/13320,create_issue,created,https://github.com/githubnext/gh-aw-cao/issues/13320,2026-09-22T01:50:55.000Z
30,dependabot,dependabot-update-planner,worker,github/gh-aw-firewall,https://github.com/githubnext/gh-aw-cao/issues/13317,create_issue,created,https://github.com/githubnext/gh-aw-cao/issues/13317,2026-09-22T01:50:55.000Z
31,dependabot,dependabot-update-planner,worker,github/gh-aw-firewall,https://github.com/githubnext/gh-aw-cao/issues/13318,create_issue,created,https://github.com/githubnext/gh-aw-cao/issues/13318,2026-09-22T01:50:55.000Z


### Native operational-value measurements

No finite operational-value measurement is available. Inspect the `value_measurement`, `grader_error`, and `grader_unavailable` columns above.

### Interpretation

- **2** partition(s) produced output evidence.
- **0** partition(s) have finite native operational-value measurements.
- **1** partition(s) have value-evaluation errors.
- Produced outputs remain **unverified production** until acceptance or disposition evidence is available.
- Use AIC and duration only as efficiency context; do not interpret lower cost as higher value by itself.

## Interpretation boundary

- `production=produced` means a successful Run created output evidence. It does not establish acceptance, implementation, usefulness, or operational value.
- `value_measurement=measured` means a native finite metric exists. Interpret it only with its native unit, direction, and campaign definition.
- `evaluation-error`, `unavailable`, and `not-configured` preserve why value cannot currently be assessed.
- AIC and duration provide efficiency context and remain separate from value.
- `observed-extra` targets remain visible but do not determine configured campaign value.
- An explicit historical analysis may examine successful Runs from a currently failed partition, but it must be labeled historical and must not replace failure diagnosis.
